# 稀疏矩阵 / Sparse Matrices

---

数组和矩阵是数值计算的基础元素。目前为止，我们都是使用NumPy的ndarray数据结构来表示数组，这是一种同构的容器，用于存储数组的所有元素。

Arrays and matrices are the fundamental elements of numerical computation. Up to now, we have been using NumPy's ndarray data structure to represent arrays, which is a homogeneous container that stores all the elements of the array.

有一种特殊情况，矩阵的大部分元素都为零，这种矩阵被称为稀疏矩阵。对于稀疏矩阵，将所有零保存在计算机内存中的效率很低，更合适的方法是只保存非零元素以及位置信息。

There is a special case in which most elements of the matrix are zero; such a matrix is called a sparse matrix. For a sparse matrix, storing all the zeros in computer memory is inefficient, and it is more appropriate to store only the non-zero elements along with their position information.

假设存在一个神经网络，16个神经元分布在一个4×4的二维矩形网格上，其中只有最近邻的神经元是相连的。那么，神经网络的连接情况就可以表示为一个稀疏矩阵。

Suppose there is a neural network with 16 neurons arranged on a 4×4 two-dimensional rectangular grid, where only nearest-neighbor neurons are connected. Then the connectivity of this neural network can be represented as a sparse matrix.

<img src="Images/Sparse_Matrix.png" style="width: 400px;"/>

<!-- bilingual -->

## 导入模块 / Importing Modules

---

SciPy中提供了稀疏矩阵模块`scipy.sparse`，为稀疏矩阵的表示及其线性代数运算提供了丰富易用的接口。

SciPy provides the sparse matrix module `scipy.sparse`, which offers a rich and easy-to-use interface for representing sparse matrices and performing their linear algebra operations.

<!-- bilingual -->

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

import numpy as np

import scipy.sparse as sp
import scipy.sparse.linalg

import scipy.linalg as la

In [ ]:
%reload_ext version_information
%version_information numpy, matplotlib, scipy, sympy

## SciPy中的稀疏矩阵 / Sparse Matrices in SciPy

---

下方表格总结和比较了了SciPy的sparse模块中可用的稀疏矩阵表示法。

The table below summarizes and compares the sparse matrix representations available in SciPy's sparse module.

| 类型 | Python类 | 描述 | 优点 | 缺点 |
|---|---|---|---|---|
| 坐标的列表 |[sp.coo_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.coo_matrix.html)| 将非零值及其行列信息保存在一个列表 | 构造简单，添加元素方便 | 访问元素效率低下 |
| 列表的列表 |[sp.lil_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.lil_matrix.html)| 将每行的非零元素列索引保存在一个列表，将对应值保存在另一个列表 | 支持切片操作 | 不方便进行数学运算 |
| 值的字典 |[sp.dok_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.dok_matrix.html)| 将非零值保存在字典，非零值的坐标元组作为字典的键 | 构造简单，可快速添加删除元素 | 不方便进行数学运算 |
| 对角矩阵 |[sp.dia_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.dia_matrix.html)| 矩阵的对角线列表 | 对于对角矩阵非常有效 | 不适用非对角矩阵 |
| 压缩列格式和压缩行格式 |[sp.csc_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csc_matrix.html) 和 [sp.csr_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.html) | 将值与行列索引的数组一起存储 | 对于矩阵的向量乘法很高效 | 构造相对复杂 |
| 块稀疏矩阵 |[sp.bsr_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.bsr_matrix.html)| 与CSR类似，用于具有稠密子矩阵的稀疏矩阵 | 对于此类特殊矩阵很高效 | 不适用一般矩阵 |

| Type | Python class | Description | Advantages | Disadvantages |
|---|---|---|---|---|
| List of coordinates |[sp.coo_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.coo_matrix.html)| Stores the non-zero values and their row/column information in a list | Simple to construct; easy to add elements | Inefficient element access |
| List of lists |[sp.lil_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.lil_matrix.html)| Stores the column indices of non-zero elements of each row in one list, and the corresponding values in another list | Supports slicing | Inconvenient for math operations |
| Dictionary of values |[sp.dok_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.dok_matrix.html)| Stores the non-zero values in a dictionary, with the coordinate tuples of the non-zero values as keys | Simple to construct; fast element insertion and deletion | Inconvenient for math operations |
| Diagonal matrix |[sp.dia_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.dia_matrix.html)| List of diagonals of the matrix | Very efficient for diagonal matrices | Not suitable for non-diagonal matrices |
| Compressed Column and Row formats |[sp.csc_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csc_matrix.html) and [sp.csr_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.html) | Stores values together with row/column index arrays | Very efficient for matrix-vector multiplication | Relatively complicated to construct |
| Block sparse matrix |[sp.bsr_matrix](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.bsr_matrix.html)| Similar to CSR; used for sparse matrices with dense submatrices | Very efficient for this kind of special matrix | Not suitable for general matrices |

<!-- bilingual -->

### 坐标列表格式 COO / Coordinate List Format (COO)

例如，我们在SciPy对下面稀疏矩阵以COO格式进行初始化：

For example, we initialize the following sparse matrix in COO format in SciPy:

$$
A = 
\begin{bmatrix}0 \ 1 \ 0 \ 0 \\
0 \ 0 \ 0 \ 2 \\
0 \ 0 \ 3 \ 0 \\
4 \ 0 \ 0 \ 0 \end{bmatrix}
$$

<!-- bilingual -->

为了创建`sp.coo_matrix`对象，我们需要创建非零值、行索引以及列索引的列表或数组，并将其传递给生成函数`sp.coo_matrix`。

To create a `sp.coo_matrix` object, we need to create lists or arrays of non-zero values, row indices, and column indices, and pass them to the constructor function `sp.coo_matrix`.

<!-- bilingual -->

In [ ]:
values = [1, 2, 3, 4]
rows = [0, 1, 2, 3]
cols = [1, 3, 2, 0]
A = sp.coo_matrix((values, (rows, cols)), shape=[4, 4])
A

SciPy的sparse模块中稀疏矩阵的属性大部分派生自NumPy的ndarray对象，同时也包括nnz（非零元素数目）和data（非零值）等属性。

Most attributes of sparse matrices in SciPy's sparse module are derived from NumPy's ndarray object, and also include attributes such as nnz (the number of non-zero elements) and data (the non-zero values).

<!-- bilingual -->

In [ ]:
A.shape, A.size, A.dtype, A.ndim

In [ ]:
A.nnz, A.data

对于`sp.coo_matrix`对象，我们还可以使用row和col属性来访问底层的行列坐标数组。

For a `sp.coo_matrix` object, we can also use the row and col attributes to access the underlying row and column coordinate arrays.

<!-- bilingual -->

In [ ]:
A.row, A.col

### 格式转换 / Format Conversion

稠密矩阵和稀疏矩阵，以及不同格式稀疏矩阵之间，可以很方便地进行格式转换。

It is easy to convert between dense and sparse matrices, as well as between different sparse matrix formats.

<!-- bilingual -->

In [ ]:
A.todense()  # 转换为稠密矩阵

In [ ]:
A.toarray()  # 转换为ndarray数组

In [ ]:
A.tocsr()  # 转换为CSR格式

主要注意，不是所有的格式支持索引。

Note that not all formats support indexing.

<!-- bilingual -->

In [ ]:
try:
    A.tobsr()[1]  # 块稀疏矩阵不支持索引
except NotImplementedError:
    print("NotImplementedError")

### 压缩列格式和压缩行格式 CSR/CSC / Compressed Column and Compressed Row Formats (CSR/CSC)

对于数值计算而言，SciPy的sparse模块中最重要的稀疏矩阵表示是CSR和CSC格式，因为它们非常适合进行有效的矩阵运算和线性代数运算。在准备好需要计算的稀疏矩阵后，可以使用`toscr`或者`tocsc`将其转换为CSR格式或者CSC格式。

For numerical computation, the most important sparse matrix representations in SciPy's sparse module are the CSR and CSC formats, because they are well suited for efficient matrix and linear algebra operations. Once we have prepared the sparse matrix we want to compute with, we can use `tocsr` or `tocsc` to convert it into the CSR or CSC format.

<!-- bilingual -->

In [ ]:
A = np.array([[1, 2, 0, 0], [0, 3, 4, 0], [0, 0, 5, 6], [7, 0, 8, 9]])
A

In [ ]:
A = sp.csr_matrix(A)
A.data  # 所有非零元素

In [ ]:
A.indices # 非零元素列序号

In [ ]:
A.indptr  # 每一行第一个非零元素在data中的起始序号

![](Images/CSR.png)

<!-- bilingual -->

## 创建稀疏矩阵 / Creating Sparse Matrices

---

一种创建稀疏矩阵的方法是对普通矩阵进行格式转换。同时，`sp.sparse`模块提供了很多用于生成此类矩阵的函数。

One way to create a sparse matrix is by converting a regular matrix. At the same time, the `sp.sparse` module provides many functions for generating such matrices.

`sp.eye`用于生成对焦矩阵，`sp.kron`用于计算两个稀疏矩阵的Kronecker张量积，`bmat`、`vstack`和`hstack`用于从稀疏块矩阵、垂直和水平堆叠矩阵来生成稀疏矩阵。

`sp.eye` is used to generate diagonal matrices, `sp.kron` is used to compute the Kronecker tensor product of two sparse matrices, and `bmat`, `vstack`, and `hstack` are used to generate a sparse matrix by stacking sparse block matrices, or by vertical or horizontal stacking.

例如，我们重复调用`sp.eye`生成一个具有多个对角线的矩阵，其中使用了k参数设置相对主对角线的偏移量。

For example, we repeatedly call `sp.eye` to generate a matrix with multiple diagonals, using the k parameter to set the offset relative to the main diagonal.

<!-- bilingual -->

In [ ]:
N = 10
A = -2 * sp.eye(N) + sp.eye(N, k=1) + sp.eye(N, k=-1)
print(type(A))
A.todense()

默认情况下，得到的是CSR格式的稀疏矩阵。使用format参数，可以指定任意其它稀疏矩阵格式。

By default, we obtain a sparse matrix in CSR format. Any other sparse matrix format can be specified using the format parameter.

<!-- bilingual -->

In [ ]:
A = sp.diags([1,-2,1], [1,0,-1], shape=[N, N], format='csc')
A

可以注意到，这个矩阵是离散空间下二阶微分操作（[拉普拉斯算子](https://zh.wikipedia.org/wiki/拉普拉斯算子)）的矩阵表示。

Note that this matrix is the matrix representation of the second-order differential operator ([Laplacian](https://zh.wikipedia.org/wiki/拉普拉斯算子)) in discrete space.

$$\frac{d^{2} f}{d x^{2}}=\lim _{h \rightarrow 0} \frac{f(x-h)-2 f(x)+f(x+h)}{h^{2}}$$

$$
\frac{d^{2} f}{d x^{2}}=\begin{bmatrix}\begin{array}{ccccccc}-2 & 1 & 0  & \ldots & 0 & 0 \\ 1 & -2 & 1 & \ldots & 0 & 0 \\ 0 & 1 & -2 & \ldots & 0 & 0 \\ \ldots & \ldots & \ldots & \ldots & \ldots & \ldots \\ 0 & 0 & 0 & \ldots & -2 & 1 \\ 0 & 0 & 0 & \ldots & 1 & -2\end{array}\end{bmatrix}\begin{bmatrix}\begin{array}{c}f(a) \\ f(a+h) \\ f(a+2 h) \\ \ldots \\ f(a+(n-2) h) \\ f(b)\end{array}\end{bmatrix} / h^{2}=\operatorname{Lap}|f\rangle
$$

<!-- bilingual -->

### 稀疏矩阵的可视化 / Visualizing Sparse Matrices

matplotlib提供的`spy`函数可以对稀疏矩阵进行可视化。

The `spy` function provided by matplotlib can be used to visualize sparse matrices.

<!-- bilingual -->

In [ ]:
fig, ax = plt.subplots()
ax.spy(A)

稀疏矩阵经常和张量积空间相关。使用`sp.kron`函数，可以将多个较小的矩阵组成单个大的稀疏矩阵。

Sparse matrices are often related to tensor-product spaces. Using the `sp.kron` function, several smaller matrices can be combined into a single large sparse matrix.

神经网络中的变换都可以简化为数值数据张量上的一些张量计算。

The transformations inside a neural network can all be simplified into some tensor operations on tensors of numerical data.

<!-- bilingual -->

In [ ]:
B = sp.diags([1, 1], [-1, 1], shape=[3,3])
B.todense()

In [ ]:
C = sp.kron(A, B, format='csr')

fig, (ax_A, ax_B, ax_C) = plt.subplots(1, 3, figsize=(12, 4))
ax_A.spy(A)
ax_B.spy(B)
ax_C.spy(C)

## 稀疏矩阵线性代数 / Sparse Linear Algebra

---

稀疏矩阵的主要应用是在大型矩阵上进行线性代数运算。SciPy的`sparse`模块中包含的`linalg`子模块实现了很多线性代数运算。

The main application of sparse matrices is linear algebra on large matrices. The `linalg` submodule inside SciPy's `sparse` module implements many linear algebra operations.

稀疏线性代数模块`scipy.sparse.linalg`和稠密线性代数模块`scipy.linalg`的运算行为存在很多差异。例如，用于稠密问题的特征值求解器通常会计算返回所有的特征值和特征向量。对于稀疏矩阵而言，为了保证稀疏性和计算效率，它的特征值求解器通常只返回少量的特征值和特征向量，例如特征值最大或最小的部分。

The sparse linear algebra module `scipy.sparse.linalg` and the dense linear algebra module `scipy.linalg` behave quite differently in many respects. For example, eigenvalue solvers for dense problems usually compute and return all eigenvalues and eigenvectors. For sparse matrices, in order to preserve sparsity and computational efficiency, the eigenvalue solver typically only returns a small number of eigenvalues and eigenvectors, such as those with the largest or smallest eigenvalues.

当然，这种差异性在大部分场景不会影响稀疏线性代数模块的应用。因为很多时候我们只关心最小或者做大的部分特征值。例如求解薛定谔方程时，特征值（能量值）最低的结果是最重要的。

Of course, this difference does not affect the applications of the sparse linear algebra module in most situations, because we often only care about the smallest or largest portion of the eigenvalues. For example, when solving the Schrödinger equation, the eigenvalue (energy) result with the lowest value is the most important.

<!-- bilingual -->

### 线性方程组 / Systems of Linear Equations

我们考虑$Ax=b$形式的线性方程组，以前面提到的三对角矩阵为例：

We consider a linear system of the form $Ax=b$, using the tridiagonal matrix mentioned earlier as an example:

<!-- bilingual -->

In [ ]:
N = 10
A = sp.diags([1, -2, 1], [1, 0, -1], shape=[N, N], format='csc')
b = -np.ones(N)

首先使用SciPy提供的稀疏矩阵求解器：

First, using the sparse matrix solver provided by SciPy:

<!-- bilingual -->

In [ ]:
x = sp.linalg.spsolve(A, b)
x

为了对比，我们也使用SciPy提供的稠密矩阵求解器：

For comparison, we also use SciPy's dense matrix solver:

<!-- bilingual -->

In [ ]:
B = A.todense()
x = la.solve(B, b)
x

为了对比两者的速度差异，我们可以尝试较大的矩阵：

To compare the speed difference, we can try a larger matrix:

<!-- bilingual -->

In [ ]:
N = 1000
A = sp.diags([1, -2, 1], [1, 0, -1], shape=[N, N], format='csc')
b = -np.ones(N)
B = A.todense()

In [ ]:
%time x = sp.linalg.spsolve(A, b)

In [ ]:
%time x = la.solve(B, b)

可以看到，当N很大时，稀疏矩阵的运算速度有很大优势。

As we can see, when N is large, the sparse matrix operation has a large speed advantage.

<!-- bilingual -->

In [ ]:
# compare performance of solving Ax=b vs system size N,
# where A is the sparse matrix for the 1d poisson problem

import time

def setup(N):
    A = sp.diags([1,-2,1], [1,0,-1], shape=[N, N], format='csr')
    b = -np.ones(N)
    return A, A.todense(), b

reps = 10
N_vec = np.arange(2, 300, 1)
t_sparse = np.empty(len(N_vec))
t_dense = np.empty(len(N_vec))
for idx, N in enumerate(N_vec):
    A, A_dense, b = setup(N)
    t = time.time()
    for r in range(reps):
        x = np.linalg.solve(A_dense, b)
    t_dense[idx] = (time.time() - t)/reps
    t = time.time()
    for r in range(reps):
        x = sp.linalg.spsolve(A, b, use_umfpack=True)
    t_sparse[idx] = (time.time() - t)/reps
    
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(N_vec, t_dense * 1e3, '.-', label="dense")
ax.plot(N_vec, t_sparse * 1e3, '.-', label="sparse")
ax.set_xlabel(r"$N$", fontsize=16)
ax.set_ylabel("elapsed time (ms)", fontsize=16)
ax.legend(loc=0)
fig.tight_layout()

### LU分解 / LU Decomposition

一种替代`spsolve`接口的方法时使用`sp.sparse.splu`或`sp.sparse.splu`（不完全LU分解）。如果需要为多组向量$b$求解$Ax=b$，这将特别有用。

One alternative to the `spsolve` interface is `sp.sparse.splu` or `sp.sparse.splu` (incomplete LU decomposition). This is particularly useful when you need to solve $Ax=b$ for multiple right-hand side vectors $b$.

<!-- bilingual -->

In [ ]:
N = 1000
A = sp.diags([1, -2, 1], [1, 0, -1], shape=[N, N], format='csc')
b = -np.ones(N)

%time lu = sp.linalg.splu(A)

In [ ]:
lu.L, lu.U

进行LU分解之后，就可以使用lu对象的solve方法有效求解$Ax=b$了。

After performing the LU decomposition, we can efficiently solve $Ax=b$ using the solve method of the lu object.

<!-- bilingual -->

In [ ]:
x = lu.solve(b)

### 特征值问题 / Eigenvalue Problems

可以分别使用`sp.linalg.eigs`和`sp.linalg.svds`函数来求解稀疏矩阵的特征值和奇异值问题。对于实数对阵矩阵或者复数Hermitian矩阵，也可以使用`sp.linalg.eigsh`函数来计算特征值和特征向量。

We can use the `sp.linalg.eigs` and `sp.linalg.svds` functions respectively to solve the eigenvalue and singular value problems of sparse matrices. For real symmetric matrices or complex Hermitian matrices, we can also use the `sp.linalg.eigsh` function to compute the eigenvalues and eigenvectors.

这些函数只会计算给定数量的特征值和特征向量（默认为6个）。使用关键词k，可以设置所需计算的特征值的数量。使用关键词which（例如，'LM'为绝对值最大，'SM'为绝对值最大），可以设置计算哪一部分的特征值。

These functions only compute a given number of eigenvalues and eigenvectors (6 by default). The keyword k can be used to set the number of eigenvalues to compute. The keyword which (e.g., 'LM' for largest in absolute value, 'SM' for smallest in absolute value) can be used to choose which portion of the eigenvalues to compute.

<!-- bilingual -->

例如，我们计算一维泊松问题（系统尺寸为10×10）中稀疏矩阵的最大模的四个特征值：

For example, we compute the four eigenvalues with the largest modulus of the sparse matrix in the one-dimensional Poisson problem (with a system size of 10×10):

<!-- bilingual -->

In [ ]:
N = 10
A = sp.diags([1, -2, 1], [1, 0, -1], shape=[N, N], format='csc')

evals, evecs = sp.linalg.eigs(A, k=4, which='LM')
evals

`sp.linalg.eigs`函数返回的是一个元组，其中第一个元素是特征值数组，第二个元素是特征向量数组。

The `sp.linalg.eigs` function returns a tuple, whose first element is an array of eigenvalues and whose second element is an array of eigenvectors.

我们期望A和特征向量之间的点积等于对应特征值对特征向量的缩放，下面将验证这一点：

We expect the dot product of A with an eigenvector to equal the eigenvector scaled by the corresponding eigenvalue. Let us verify this below:

<!-- bilingual -->

In [ ]:
np.allclose(A.dot(evecs[:,0]), evals[0] * evecs[:,0])  # 排除浮点数误差影响

我们下面将对比稀疏矩阵和稠密矩阵在计算大矩阵特征值问题上的速度差异：

Below we will compare the speed difference between sparse and dense matrices when computing the eigenvalue problem of a large matrix:

<!-- bilingual -->

In [ ]:
N = 1000
A = sp.diags([1, -2, 1], [1, 0, -1], shape=[N, N], format='csc')
B = A.todense()

In [ ]:
%time evals, evecs = sp.linalg.eigs(A, k=4, which='LM')

In [ ]:
%time evals, evecs = la.eig(B)